# Dataset and DataLoader in PyTorch

In [9]:
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from PIL import Image
from typing import Optional
import plotly.graph_objects as go

## Dataset

In [10]:
class CustomImageDataset(Dataset):
    def __init__(self, data_root: str, transform: Optional[callable] = None):
        super().__init__()

        self.label_dict = {
            "astilbe": 0,
            "bellflower": 1,
            "black_eyed_susan": 2,
            "calendula": 3,
            "california_poppy": 4,
            "carnation": 5,
            "common_daisy": 6,
            "coreopsis": 7,
            "daffodil": 8,
            "dandelion": 9,
            "iris": 10,
            "magnolia": 11,
            "rose": 12,
            "sunflower": 13,
            "tulip": 14,
            "water_lily": 15,
        }
        self.data_root = data_root
        self.transform = transform
        self.image_paths, self.labels = self._get_data()

    def _get_data(self):
        label_dirs = os.listdir(self.data_root)
        image_paths = []
        labels = []
        for label_dir in label_dirs:
            images = glob.glob(os.path.join(self.data_root, label_dir, "*.jpg"))
            image_paths.extend(images)
            labels.extend([self.label_dict[label_dir]] * len(images))
        return image_paths, labels

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx): # abstract method
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

## DataLoader

In [11]:
transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
dataset = CustomImageDataset(data_root=r"D:\DataSets\CLS\flowers", transform=transforms)

In [12]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)

In [13]:
# visualize some images
images, labels = next(iter(dataloader))
print(images.shape, labels.shape)
image, label = images[0], labels[0]
image = torch.clip(image, 0, 1).permute(1, 2, 0).numpy() * 255
fig = go.Figure(data=[go.Image(z=image)])
fig.update_layout(title=f"Label: {label}")
fig.show()

torch.Size([32, 3, 224, 224]) torch.Size([32])


In [14]:
# custom sampler
class CustomSampler(Sampler):
    def __init__(self, data_source):
        super().__init__()
        self.data_source = data_source

    def __iter__(self):
        return iter(torch.randperm(len(self.data_source)).tolist())

    def __len__(self):
        return len(self.data_source)

In [15]:
# custom collate_fn
def custom_collate_fn(batch):
    images, labels = zip(*batch)
    images = torch.stack(images)
    labels = torch.tensor(labels)
    return images, labels